In [5]:
print("1- Importations et chargement des données \n")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline

# Chargement du train set prétraité
train_df = pd.read_csv("../data/processed/train_preprocessed.csv")

print("Dimensions du train set :", train_df.shape)
print("Distribution de la cible (stroke) :")
print(train_df["stroke"].value_counts(normalize=True) * 100)

1- Importations et chargement des données 

Dimensions du train set : (4087, 17)
Distribution de la cible (stroke) :
stroke
0    95.130903
1     4.869097
Name: proportion, dtype: float64


In [6]:
print("2- Séparation des caractéristiques et de la cible\n ")


X_train = train_df.drop(columns=["stroke"])
y_train = train_df["stroke"]

print("Variables explicatives :", list(X_train.columns))
print("Nombre de caractéristiques :", X_train.shape[1])

2- Séparation des caractéristiques et de la cible
 
Variables explicatives : ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'Residence_type', 'avg_glucose_level', 'bmi', 'bmi_missing', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']
Nombre de caractéristiques : 16


In [7]:
print("3- Configuration de la validation croisée et des métriques\n")

# Utilisation d'une validation croisée stratifiée pour respecter le déséquilibre de classe
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Définition des métriques à évaluer
scoring = ['recall', 'precision', 'average_precision', 'roc_auc']

3- Configuration de la validation croisée et des métriques



In [9]:
print("4- Définition et évaluation des modèles\n")

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Dictionnaire complet incluant tous les modèles validés pour le projet
models = {
    # 1. Régression Logistique
    "LogisticRegression (aucune)": LogisticRegression(random_state=42),
    "LogisticRegression (class_weight)": LogisticRegression(class_weight='balanced', random_state=42),
    "LogisticRegression (SMOTE)": make_pipeline(SMOTE(random_state=42), LogisticRegression(random_state=42)),
    
    # 2. K-Plus Proches Voisins (KNN)
    "KNN (aucun)": KNeighborsClassifier(),
    "KNN (SMOTE)": make_pipeline(SMOTE(random_state=42), KNeighborsClassifier()),
    
    # 3. Arbre de Décision
    "DecisionTree (aucune)": DecisionTreeClassifier(random_state=42),
    "DecisionTree (class_weight)": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    
    # 4. Random Forest
    "RandomForest (aucune)": RandomForestClassifier(random_state=42),
    "RandomForest (class_weight)": RandomForestClassifier(class_weight='balanced', random_state=42),
    "RandomForest (SMOTE)": make_pipeline(SMOTE(random_state=42), RandomForestClassifier(random_state=42)),
    
    # 5. XGBoost
    "XGBoost (aucune)": XGBClassifier(random_state=42),
    "XGBoost (class_weight)": XGBClassifier(scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train), random_state=42)
}

results = []

for name, model in models.items():
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    results.append({
        "Modèle": name,
        "Recall": cv_results['test_recall'].mean(),
        "Precision": cv_results['test_precision'].mean(),
        "PR-AUC": cv_results['test_average_precision'].mean(),
        "ROC-AUC": cv_results['test_roc_auc'].mean()
    })

results_df = pd.DataFrame(results)
# Tri par PR-AUC décroissant pour voir les meilleurs compromis sur données déséquilibrées
results_df = results_df.sort_values(by="PR-AUC", ascending=False).reset_index(drop=True)
display(results_df)

4- Définition et évaluation des modèles



,Modèle,Recall,Precision,PR-AUC,ROC-AUC
0,LogisticRegression (aucune),0.020128,0.266667,0.227259,0.852870
1,LogisticRegression (class_weight),0.799231,0.142029,0.224562,0.853658
2,RandomForest (aucune),0.040000,0.380000,0.208439,0.822043
3,XGBoost (class_weight),0.190641,0.216741,0.188091,0.800452
4,RandomForest (class_weight),0.130641,0.255657,0.186215,0.825325
5,LogisticRegression (SMOTE),0.683590,0.130113,0.182464,0.816256
6,XGBoost (aucune),0.090385,0.271062,0.175106,0.805704
7,RandomForest (SMOTE),0.236538,0.135515,0.138447,0.798460
8,KNN (aucun),0.020000,0.196667,0.092162,0.634366
9,KNN (SMOTE),0.432949,0.105423,0.088890,0.686310


In [10]:
print("5- Sélection des meilleurs modèles pour l'évaluation finale\n")

# Exemple : Choisir les deux meilleurs modèles d'après le tableau de résultats du benchmark
CHOICES = [
    ("RandomForest", "class_weight"),
    ("XGBoost", "class_weight"),
]

print("Modèles sélectionnés pour l'évaluation finale (04_evaluation) :")
for model, strategy in CHOICES:
    print(f"- {model} avec la stratégie {strategy}")

5- Sélection des meilleurs modèles pour l'évaluation finale

Modèles sélectionnés pour l'évaluation finale (04_evaluation) :
- RandomForest avec la stratégie class_weight
- XGBoost avec la stratégie class_weight
